# Murder Mystery LLM Server Control

Use this notebook to start and stop the FastAPI server from Jupyter.

Prereqs:
- Install dependencies (see `pyproject.toml`).
- Run this notebook from the repo root so `src/` is in the expected location.


In [1]:
import os
import sys
import time
from pathlib import Path
from urllib.request import urlopen
from urllib.error import URLError
import subprocess

APP = "mmllm.web.main:app"
HOST = "127.0.0.1"
PORT = 8000
HEALTH_URL = f"http://{HOST}:{PORT}/health"

SERVER_PROCESS = None
SERVER_LOG = None
SERVER_LOG_PATH = None

def _env_with_pythonpath():
    env = os.environ.copy()
    repo_root = Path.cwd()
    src_path = repo_root / "src"
    env["PYTHONPATH"] = str(src_path)
    return env


In [16]:
def start_server():
    global SERVER_PROCESS, SERVER_LOG, SERVER_LOG_PATH
    if SERVER_PROCESS and SERVER_PROCESS.poll() is None:
        print("Server already running.")
        return

    repo_root = Path.cwd()
    if not (repo_root / "src").exists() and (repo_root.parent / "src").exists():
        repo_root = repo_root.parent
    print(f"Starting server from: {repo_root}")

    env = _env_with_pythonpath()
    env["PYTHONUNBUFFERED"] = "1"
    log_dir = repo_root / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    SERVER_LOG_PATH = log_dir / "server.log"
    SERVER_LOG = open(SERVER_LOG_PATH, "a", encoding="utf-8", buffering=1)
    SERVER_LOG.write(f"\n=== Server start {time.strftime('%Y-%m-%d %H:%M:%S')} ===\n")
    SERVER_LOG.flush()
    print(f"Logging to: {SERVER_LOG_PATH}")
    cmd = [
        sys.executable,
        "-m",
        "uvicorn",
        APP,
        "--host",
        HOST,
        "--port",
        str(PORT),
        "--log-level",
        "debug",
        "--access-log",
    ]
    print("Command:", " ".join(cmd))
    SERVER_PROCESS = subprocess.Popen(
        cmd,
        env=env,
        cwd=repo_root,
        stdout=SERVER_LOG,
        stderr=subprocess.STDOUT,
    )
    print(f"Server process pid: {SERVER_PROCESS.pid}")

    for _ in range(60):
        if SERVER_PROCESS.poll() is not None:
            print("Server exited early.")
            if SERVER_LOG:
                SERVER_LOG.flush()
                SERVER_LOG.close()
                SERVER_LOG = None
            if SERVER_LOG_PATH:
                print(f"See log at: {SERVER_LOG_PATH}")
            raise RuntimeError("Server exited early.")
        try:
            with urlopen(HEALTH_URL, timeout=1) as resp:
                if resp.status == 200:
                    print("Server is up.")
                    return
        except URLError:
            time.sleep(0.25)
    print("Server started, but health check did not respond yet.")

start_server()


Starting server from: d:\Source\murder-mystery-llm
Logging to: d:\Source\murder-mystery-llm\logs\server.log
Command: d:\Source\murder-mystery-llm\.venv\Scripts\python.exe -m uvicorn mmllm.web.main:app --host 127.0.0.1 --port 8000 --log-level debug --access-log
Server process pid: 27112
Server is up.


In [17]:
def stop_server():
    global SERVER_PROCESS, SERVER_LOG, SERVER_LOG_PATH
    if not SERVER_PROCESS or SERVER_PROCESS.poll() is not None:
        print("Server is not running.")
        if SERVER_LOG:
            SERVER_LOG.close()
        SERVER_LOG = None
        SERVER_LOG_PATH = None
        SERVER_PROCESS = None
        return

    SERVER_PROCESS.terminate()
    try:
        SERVER_PROCESS.wait(timeout=5)
        print("Server stopped.")
    except subprocess.TimeoutExpired:
        SERVER_PROCESS.kill()
        print("Server killed after timeout.")
    finally:
        if SERVER_LOG:
            SERVER_LOG.write(f"\n=== Server stop {time.strftime('%Y-%m-%d %H:%M:%S')} ===\n")
            SERVER_LOG.flush()
            SERVER_LOG.close()
        SERVER_LOG = None
        SERVER_LOG_PATH = None
        SERVER_PROCESS = None

stop_server()


Server stopped.


In [13]:
def check_health():
    if SERVER_PROCESS is None:
        print("Server process not started.")
    else:
        status = SERVER_PROCESS.poll()
        print(f"Server process poll: {status}")
        if status is not None:
            try:
                stdout, stderr = SERVER_PROCESS.communicate(timeout=1)
                print("STDOUT:")
                print(stdout or "<empty>")
                print("STDERR:")
                print(stderr or "<empty>")
            except subprocess.TimeoutExpired:
                print("Process exited but output not available yet.")
    try:
        with urlopen(HEALTH_URL, timeout=2) as resp:
            body = resp.read().decode("utf-8", errors="replace")
            print("/health", resp.status, body)
    except URLError as exc:
        print(f"Health check failed: {exc}")
    try:
        with urlopen(f"http://{HOST}:{PORT}/", timeout=2) as resp:
            body = resp.read().decode("utf-8", errors="replace")
            print("/", resp.status, body[:200])
    except URLError as exc:
        print(f"Root check failed: {exc}")

check_health()


Server process not started.
Health check failed: <urlopen error timed out>
Root check failed: <urlopen error timed out>
